# AE4: integración Steam–Twitch

Notebook autónomo para ejecutar el ETL de los dos CSV del proyecto. Las funciones de ingesta, limpieza, cruce y validación están incluidas en este archivo y no dependen de un script Python externo.

## Decisiones analíticas
- P1: se reporta aparte la ausencia de aparición observada en el top 200. Los cuatro cuartiles se calculan entre juegos con `horas_3m > 0`; la ausencia equivale a visibilidad por debajo del umbral observado, no a cero transmisiones.
- P4: `streamers_max_3m` es el máximo conteo mensual de streamers, no personas únicas. Twitch no ofrece identificadores individuales en esta fuente.
- Steam `Price` ya está en USD decimales; no se divide por 100.

Ejecutar las celdas en orden. Los CSV procesados y `validaciones.txt` se guardan en `salida_etl/`.

In [3]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

BASE_DIR = Path.cwd()
if not (BASE_DIR / "games.csv").exists() and (BASE_DIR.parent / "games.csv").exists():
    BASE_DIR = BASE_DIR.parent

RUTA_STEAM = BASE_DIR / "games.csv"
RUTA_TWITCH = BASE_DIR / "Twitch_game_data.csv"
SALIDA = BASE_DIR / "salida_etl"
SALIDA.mkdir(parents=True, exist_ok=True)

for ruta in (RUTA_STEAM, RUTA_TWITCH):
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido: {ruta}")

VENTANA_DESDE = pd.Timestamp("2016-01-01")
VENTANA_HASTA = pd.Timestamp("2024-06-30")
REPORTE = []

COLS_STEAM = [
    "AppID", "Name", "Release date", "Estimated owners", "Peak CCU", "Required age",
    "Price", "Discount", "DLC count", "About the game", "Supported languages",
    "Full audio languages", "Reviews", "Header image", "Website", "Support url",
    "Support email", "Windows", "Mac", "Linux", "Metacritic score", "Metacritic url",
    "User score", "Positive", "Negative", "Score rank", "Achievements",
    "Recommendations", "Notes", "Average playtime forever",
    "Average playtime two weeks", "Median playtime forever",
    "Median playtime two weeks", "Developers", "Publishers", "Categories",
    "Genres", "Tags", "Screenshots", "Movies",
]

NO_JUEGOS = {
    "justchatting", "slots", "music", "specialevents", "sports", "irl",
    "talkshowspodcasts", "asmr", "art", "poker", "virtualcasino",
    "poolshottubsandbeaches", "traveloutdoors", "gamesdemos", "creative",
    "casinojackpot", "fooddrink", "casino", "retro", "imonlysleeping",
    "sciencetechnology", "politics", "fitnesshealth", "makerscrafting",
    "softwareandgamedevelopment", "animalsaquariumszoos", "gambling",
    "alwayson", "watchparties", "justsleeping", "coworkingstudying",
    "musicperformingarts", "chess", "dungeonsdragons",
}

ALIAS_TWITCH_A_STEAM = {
    "playerunknownsbattlegrounds": "pubgbattlegrounds",
    "counterstrikeglobaloffensive": "counterstrike2",
    "grandtheftautov": "grandtheftautovlegacy",
    "tomclancysrainbowsixsiege": "tomclancysrainbowsixsiegex",
    "fifa21": "easportsfifa21",
}

REGEX_NO_BASE = re.compile(r"\b(?:playtest|demo|soundtrack|dedicated server)\b", re.I)


def log(msg=""):
    print(msg)
    REPORTE.append(str(msg))

print(f"Entradas: {RUTA_STEAM.name}, {RUTA_TWITCH.name}")
print(f"Salidas: {SALIDA}")

Entradas: games.csv, Twitch_game_data.csv
Salidas: c:\Users\Bangho\Desktop\ae4 base de datos\salida_etl


In [4]:
def arreglar_texto(s):
    if not isinstance(s, str):
        return s
    s = re.sub(r"<U\+([0-9A-Fa-f]{4})>", lambda match: chr(int(match.group(1), 16)), s)
    try:
        return s.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return s


def normalizar(s):
    s = re.sub(r"[™®©]", "", str(s))
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", s.lower())


def extraer():
    log("=" * 70)
    log("1. INGESTIÓN")
    log("=" * 70)
    steam = pd.read_csv(RUTA_STEAM, encoding="utf-8-sig", header=0, names=COLS_STEAM)
    twitch = pd.read_csv(RUTA_TWITCH, encoding="cp1252")
    log(f"Steam ({RUTA_STEAM.name}): {steam.shape[0]:,} filas x {steam.shape[1]} columnas")
    log(f"Twitch ({RUTA_TWITCH.name}): {twitch.shape[0]:,} filas x {twitch.shape[1]} columnas")
    assert steam.shape[1] == len(COLS_STEAM), "Steam no tiene 40 columnas"
    log("Steam: se asignaron los 40 nombres de columna esperados al leer el CSV.")
    return steam, twitch


def limpiar_steam(df):
    log("\n" + "=" * 70 + "\n2. LIMPIEZA Y TRANSFORMACIÓN - STEAM\n" + "=" * 70)
    n0 = len(df)

    n = df["Name"].isna().sum()
    df = df.dropna(subset=["Name"]).copy()
    log(f"Nulos: {n} fila(s) sin nombre eliminadas.")

    n = df.duplicated("AppID").sum()
    df = df.drop_duplicates("AppID")
    log(f"Duplicados por AppID: {n}.")

    df["AppID"] = df["AppID"].astype("int64")
    df["precio_usd"] = pd.to_numeric(df["Price"], errors="coerce")
    log("Precio: el CSV ya está expresado en dólares con decimales; se conserva el valor.")
    df["fecha_lanzamiento"] = pd.to_datetime(df["Release date"], format="mixed", errors="coerce")
    log(f"Fechas de lanzamiento no interpretables (NaT): {df['fecha_lanzamiento'].isna().sum()}.")

    rango = df["Estimated owners"].str.split(" - ", expand=True).astype(float)
    df["owners_min"], df["owners_max"] = rango[0], rango[1]
    df["owners_medio"] = (rango[0] + rango[1]) / 2
    log("Owners: se usa el punto medio del rango publicado por Steam.")

    df["nombre"] = df["Name"].map(arreglar_texto)
    log(f"Nombres con mojibake reparados: {(df['nombre'] != df['Name']).sum():,}.")

    mask = df["nombre"].str.contains(REGEX_NO_BASE, na=False)
    log(f"Registros que no son el juego base eliminados: {mask.sum():,}.")
    df = df[~mask]

    mask = df["owners_max"] <= 0
    log(f"Juegos con owners '0 - 0' excluidos: {mask.sum():,}.")
    df = df[~mask]

    df["precio_atipico"] = df["precio_usd"] > 100
    log(f"Precios mayores a 100 USD marcados: {df['precio_atipico'].sum():,}; no se eliminan.")

    df["genero_principal"] = (
        df["Genres"].fillna("").str.split(",").str[0].str.strip().replace("", "Sin género")
    )
    df["clave"] = df["nombre"].map(normalizar)
    df["n_appid_mismo_nombre"] = df.groupby("clave")["AppID"].transform("size")
    df["resenas_total"] = df["Positive"].fillna(0) + df["Negative"].fillna(0)

    antes = len(df)
    df = (df.sort_values(["owners_medio", "resenas_total"], ascending=False)
            .drop_duplicates("clave", keep="first"))
    log(f"Nombres con varios AppID: se conservaron los de mayor owners; filas eliminadas: {antes - len(df):,}.")
    df["en_ventana"] = df["fecha_lanzamiento"].between(VENTANA_DESDE, VENTANA_HASTA)

    cols = ["AppID", "nombre", "clave", "fecha_lanzamiento", "en_ventana", "precio_usd",
            "precio_atipico", "owners_min", "owners_max", "owners_medio", "Peak CCU",
            "Positive", "Negative", "resenas_total", "Recommendations", "genero_principal",
            "Genres", "Developers", "Publishers", "n_appid_mismo_nombre"]
    out = df[cols].reset_index(drop=True)
    log(f"STEAM: {n0:,} filas de entrada -> {len(out):,} filas limpias.")
    return out


def limpiar_twitch(df):
    log("\n" + "=" * 70 + "\n2. LIMPIEZA Y TRANSFORMACIÓN - TWITCH\n" + "=" * 70)
    n0 = len(df)
    n = df["Game"].isna().sum()
    df = df.dropna(subset=["Game"]).copy()
    log(f"Nulos: {n} fila(s) sin nombre de juego eliminadas.")

    assert df["Month"].between(1, 12).all(), "Mes fuera de rango"
    log(f"Rango temporal: {df['Year'].min()}-{df['Month'][df['Year'] == df['Year'].min()].min():02d} "
        f"a {df['Year'].max()}-{df['Month'][df['Year'] == df['Year'].max()].max():02d}.")
    log(f"Filas con Streamers = 0: {(df['Streamers'] == 0).sum()} (se conservan).")

    df["fecha"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1))
    df["juego"] = df["Game"].map(arreglar_texto)
    df["clave_original"] = df["juego"].map(normalizar)
    mask = df["clave_original"].isin(NO_JUEGOS)
    log(f"Categorías que no son juegos excluidas: {mask.sum():,} filas.")
    df = df[~mask]

    df["clave"] = df["clave_original"].replace(ALIAS_TWITCH_A_STEAM)
    log(f"Filas con alias manual aplicado: {(df['clave'] != df['clave_original']).sum():,}.")

    dup = df.duplicated(["clave", "fecha"]).sum()
    log(f"Duplicados juego-mes agregados: {dup}.")
    agg = {"juego": "first", "Hours_watched": "sum", "Hours_streamed": "sum",
           "Peak_viewers": "max", "Peak_channels": "max", "Streamers": "max",
           "Avg_viewers": "max", "Avg_channels": "max", "Rank": "min"}
    out = df.groupby(["clave", "fecha"], as_index=False).agg(agg).rename(columns=str.lower)
    out["fecha"] = pd.to_datetime(out["fecha"])
    assert not out.duplicated(["clave", "fecha"]).any()
    log(f"TWITCH: {n0:,} filas de entrada -> {len(out):,} filas juego-mes limpias.")
    return out


def integrar(steam, twitch):
    log("\n" + "=" * 70 + "\n4. MERGE\n" + "=" * 70)
    log("Clave de cruce: nombre normalizado y cinco alias manuales; no hay ID común.")
    assert steam["clave"].is_unique, "Steam: clave duplicada, el join duplicaría filas"

    ref = steam[["clave", "AppID", "nombre", "fecha_lanzamiento", "en_ventana",
                 "owners_medio", "genero_principal"]]
    mensual = twitch.merge(ref, on="clave", how="left", indicator=True, validate="m:1")
    mensual["cruzo"] = mensual["_merge"] == "both"
    mensual = mensual.drop(columns="_merge")

    f, l = mensual["fecha"], mensual["fecha_lanzamiento"]
    mensual["mes_rel"] = (f.dt.year - l.dt.year) * 12 + (f.dt.month - l.dt.month)
    cr = mensual[mensual["cruzo"]]
    primeros3 = cr[cr["mes_rel"].between(0, 2)]
    agg_juego = cr.groupby("clave").agg(
        horas_totales=("hours_watched", "sum"),
        meses_en_top=("fecha", "nunique"),
        primer_mes_twitch=("fecha", "min"),
        ultimo_mes_twitch=("fecha", "max"),
        pico_espectadores_max=("peak_viewers", "max"),
        streamers_max_mensual=("streamers", "max"),
    )
    p3 = primeros3.groupby("clave").agg(
        horas_3m=("hours_watched", "sum"),
        streamers_max_3m=("streamers", "max"),
        pico_espectadores_3m=("peak_viewers", "max"),
    )
    agg_juego = agg_juego.join(p3, how="left").reset_index()
    juegos = steam.merge(agg_juego, on="clave", how="left", validate="1:1")
    cols_num = ["horas_totales", "meses_en_top", "horas_3m", "streamers_max_3m",
                "pico_espectadores_3m", "pico_espectadores_max", "streamers_max_mensual"]
    juegos[cols_num] = juegos[cols_num].fillna(0)
    juegos["en_twitch"] = juegos["meses_en_top"] > 0
    lim = juegos["fecha_lanzamiento"] - pd.DateOffset(months=1)
    juegos["match_sospechoso"] = juegos["en_twitch"] & (juegos["primer_mes_twitch"] < lim)
    juegos["apto_p1"] = juegos["en_ventana"] & ~juegos["match_sospechoso"]
    log("Los juegos ausentes del top 200 reciben 0 horas, que significa bajo el umbral observado.")
    return mensual, juegos


def validar(steam_crudo_n, twitch_crudo_n, steam, twitch, mensual, juegos):
    log("\n" + "=" * 70 + "\n5. VALIDACIONES POSTERIORES AL MERGE\n" + "=" * 70)
    log(f"Steam: {steam_crudo_n:,} -> {len(steam):,} filas.")
    log(f"Twitch: {twitch_crudo_n:,} -> {len(twitch):,} filas.")
    assert len(twitch) == len(mensual) and len(steam) == len(juegos)

    cols_tw = twitch.columns.tolist()
    nulos_antes = twitch[cols_tw].isna().sum()
    nulos_despues = mensual[cols_tw].isna().sum()
    nulos_introducidos = int((nulos_despues - nulos_antes).clip(lower=0).sum())
    log(f"Nulos preexistentes en Twitch: {int(nulos_antes.sum())}.")
    log(f"Nulos introducidos en todas las columnas de Twitch: {nulos_introducidos}.")
    assert nulos_introducidos == 0, "El merge introdujo nulos en columnas de Twitch"
    assert not juegos["clave"].duplicated().any()
    assert not mensual.duplicated(["clave", "fecha"]).any()
    log("Duplicados en clave de juego y juego-mes: 0.")

    total = twitch["hours_watched"].sum()
    cruzadas = mensual.loc[mensual["cruzo"], "hours_watched"].sum()
    sin_cruce = mensual.loc[~mensual["cruzo"], "hours_watched"].sum()
    log(f"Horas conservadas: {cruzadas:,} cruzadas + {sin_cruce:,} sin cruce = {total:,} total.")
    assert cruzadas + sin_cruce == total

    cobertura = mensual.groupby("clave")["cruzo"].first()
    log(f"Cobertura por juego: {cobertura.sum():,} de {len(cobertura):,} ({cobertura.mean():.1%}).")
    log(f"Cobertura por horas: {cruzadas / total:.1%}.")
    cohorte = juegos["apto_p1"]
    positivos = cohorte & (juegos["horas_3m"] > 0)
    log(f"Cohorte P1: {int(cohorte.sum()):,}; con horas_3m positivas: {int(positivos.sum()):,}.")

    sin_match = (mensual[~mensual["cruzo"]]
                 .groupby("clave").agg(juego=("juego", "first"),
                                        horas=("hours_watched", "sum"),
                                        meses=("fecha", "nunique"))
                 .sort_values("horas", ascending=False).reset_index())
    sin_match.to_csv(SALIDA / "twitch_sin_match.csv", index=False, encoding="utf-8-sig")
    log("Títulos Twitch sin cruce (primeros 15):")
    for _, row in sin_match.head(15).iterrows():
        log(f"{row['juego']}: {row['horas']:,} horas")

## 1. Ingesta

Cada fuente se lee por separado. Steam usa sus 40 nombres de columna esperados y Twitch se decodifica como Windows-1252 según los bytes observados en el CSV.

In [5]:
steam_raw, twitch_raw = extraer()

print("Steam:", steam_raw.shape)
print("Twitch:", twitch_raw.shape)
display(steam_raw.head(3))
display(twitch_raw.head(3))

1. INGESTIÓN
Steam (games.csv): 125,855 filas x 40 columnas
Twitch (Twitch_game_data.csv): 21,000 filas x 12 columnas
Steam: se asignaron los 40 nombres de columna esperados al leer el CSV.
Steam: (125855, 40)
Twitch: (21000, 12)


,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,Discount,DLC count,About the game,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
0,2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
1,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
2,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN


,Rank,Game,Month,Year,Hours_watched,Hours_streamed,Peak_viewers,Peak_channels,Streamers,Avg_viewers,Avg_channels,Avg_viewer_ratio
0,1,League of Legends,1,2016,94377226,1362044,530270,2903,129172,127021,1833,69.29
1,2,Counter-Strike: Global Offensive,1,2016,47832863,830105,372654,2197,120849,64378,1117,57.62
2,3,Dota 2,1,2016,45185893,433397,315083,1100,44074,60815,583,104.26


## 2. Limpieza y transformación

Se limpian por separado las fuentes: se calculan owners medios y se conserva el precio en USD; Twitch se agrega a una fila por juego y mes.

In [6]:
steam = limpiar_steam(steam_raw)
twitch = limpiar_twitch(twitch_raw)

print("Steam limpio:", steam.shape)
print("Twitch limpio:", twitch.shape)
display(steam.head(3))
display(twitch.head(3))


2. LIMPIEZA Y TRANSFORMACIÓN - STEAM
Nulos: 1 fila(s) sin nombre eliminadas.
Duplicados por AppID: 0.
Precio: el CSV ya está expresado en dólares con decimales; se conserva el valor.
Fechas de lanzamiento no interpretables (NaT): 0.
Owners: se usa el punto medio del rango publicado por Steam.
Nombres con mojibake reparados: 0.
Registros que no son el juego base eliminados: 8,223.
Juegos con owners '0 - 0' excluidos: 16,404.
Precios mayores a 100 USD marcados: 188; no se eliminan.
Nombres con varios AppID: se conservaron los de mayor owners; filas eliminadas: 3,100.
STEAM: 125,855 filas de entrada -> 98,127 filas limpias.

2. LIMPIEZA Y TRANSFORMACIÓN - TWITCH
Nulos: 1 fila(s) sin nombre de juego eliminadas.
Rango temporal: 2016-01 a 2024-09.
Filas con Streamers = 0: 13 (se conservan).
Categorías que no son juegos excluidas: 1,776 filas.
Filas con alias manual aplicado: 367.
Duplicados juego-mes agregados: 11.
TWITCH: 21,000 filas de entrada -> 19,212 filas juego-mes limpias.
Steam lim

,AppID,nombre,clave,fecha_lanzamiento,en_ventana,precio_usd,precio_atipico,owners_min,owners_max,owners_medio,Peak CCU,Positive,Negative,resenas_total,Recommendations,genero_principal,Genres,Developers,Publishers,n_appid_mismo_nombre
0,730,Counter-Strike 2,counterstrike2,2012-08-21,False,0.0,False,100000000.0,200000000.0,150000000.0,1013936,7642084,1173003,8815087,4830455,Action,"Action,Free To Play",Valve,Valve,1
1,578080,PUBG: BATTLEGROUNDS,pubgbattlegrounds,2017-12-21,True,0.0,False,100000000.0,200000000.0,150000000.0,314682,1520457,1037487,2557944,1753261,Action,"Action,Adventure,Massively Multiplayer,Free To...",PUBG Corporation,"KRAFTON, Inc.",1
2,570,Dota 2,dota2,2013-07-09,False,0.0,False,100000000.0,200000000.0,150000000.0,623941,2037143,461826,2498969,14369,Action,"Action,Strategy,Free To Play",Valve,Valve,1


,clave,fecha,juego,hours_watched,hours_streamed,peak_viewers,peak_channels,streamers,avg_viewers,avg_channels,rank
0,20minutestilldawn,2022-06-01,20 Minutes Till Dawn,911356,12253,29743,80,3616,1267,17,159
1,2xko,2024-08-01,2XKO,2137305,31942,60961,508,6619,2876,42,94
2,60parsecs,2018-09-01,60 Parsecs!,529688,1867,31960,27,606,736,2,109


## 3. Cruce de fuentes y métricas

El grano mensual de Twitch se cruza con Steam por la clave normalizada del nombre. Los meses relativos 0, 1 y 2 forman la ventana de lanzamiento.

In [7]:
mensual, juegos = integrar(steam, twitch)

print("Twitch mensual unido:", mensual.shape)
print("Universo Steam con métricas Twitch:", juegos.shape)
display(juegos.loc[juegos["apto_p1"], ["nombre", "horas_3m", "owners_medio", "en_twitch"]].head(10))


4. MERGE
Clave de cruce: nombre normalizado y cinco alias manuales; no hay ID común.
Los juegos ausentes del top 200 reciben 0 horas, que significa bajo el umbral observado.
Twitch mensual unido: (19212, 19)
Universo Steam con métricas Twitch: (98127, 32)


,nombre,horas_3m,owners_medio,en_twitch
8,Call of Duty®,0.0,75000000.0,True
9,Unturned,0.0,75000000.0,False
10,Palworld,77749506.0,75000000.0,True
11,New World: Aeternum,0.0,75000000.0,True
18,HELLDIVERS™ 2,41446033.0,35000000.0,True
19,Wallpaper Engine,0.0,35000000.0,False
20,Stardew Valley,7661186.0,35000000.0,True
23,Dead by Daylight,31774600.0,35000000.0,True
27,Among Us,0.0,35000000.0,True
32,Fall Guys,0.0,35000000.0,True


## 4. Validaciones y exportación

Se verifican filas, nulos, duplicados, conservación de horas y cobertura del merge. Luego se guardan las tablas procesadas en `salida_etl/`.

In [8]:
REPORTE.clear()
validar(len(steam_raw), len(twitch_raw), steam, twitch, mensual, juegos)

steam.to_csv(SALIDA / "steam_limpio.csv", index=False, encoding="utf-8-sig")
twitch.to_csv(SALIDA / "twitch_limpio.csv", index=False, encoding="utf-8-sig")
mensual.loc[mensual["cruzo"]].to_csv(SALIDA / "twitch_mensual.csv", index=False, encoding="utf-8-sig")
juegos.to_csv(SALIDA / "juegos_merge.csv", index=False, encoding="utf-8-sig")
(SALIDA / "validaciones.txt").write_text("\n".join(REPORTE), encoding="utf-8")

print(f"Archivos exportados a: {SALIDA}")


5. VALIDACIONES POSTERIORES AL MERGE
Steam: 125,855 -> 98,127 filas.
Twitch: 21,000 -> 19,212 filas.
Nulos preexistentes en Twitch: 0.
Nulos introducidos en todas las columnas de Twitch: 0.
Duplicados en clave de juego y juego-mes: 0.
Horas conservadas: 48,825,512,791 cruzadas + 49,636,608,462 sin cruce = 98,462,121,253 total.
Cobertura por juego: 1,351 de 2,288 (59.0%).
Cobertura por horas: 49.6%.
Cohorte P1: 69,314; con horas_3m positivas: 659.
Títulos Twitch sin cruce (primeros 15):
League of Legends: 11,296,781,271 horas
Fortnite: 6,224,642,826 horas
VALORANT: 4,631,862,775 horas
Minecraft: 3,162,054,933 horas
World of Warcraft: 2,870,508,014 horas
Call of Duty: Warzone: 2,764,017,645 horas
Hearthstone: 2,513,198,715 horas
Overwatch: 1,393,597,856 horas
Teamfight Tactics: 1,169,318,581 horas
Genshin Impact: 547,266,089 horas
StarCraft II: 393,959,559 horas
Magic: The Gathering: 389,861,440 horas
Sea of Thieves: 373,694,955 horas
FIFA 23: 326,537,693 horas
New World: 277,890,359 ho